In [1]:
!nvidia-smi


Thu Oct  1 08:08:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content
!ls src data | head -20

/content
data:
code_holdout.jsonl
code_train.jsonl
code_train_v2.jsonl
code_val.jsonl
code_val_v2.jsonl
eval
json_holdout.jsonl
json_train.jsonl
json_val.jsonl
router_testset_handwritten.jsonl
router_testset.jsonl
sql_holdout.jsonl
sql_train.jsonl
sql_train_v2.jsonl
sql_val.jsonl
sql_val_v2.jsonl

src:
cascade.py


In [3]:
!pip install -q -U transformers peft "trl>=0.20" bitsandbytes accelerate datasets



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 111.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 66.8 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
%cd /content
import os; os.environ["HF_LOCAL_FILES_ONLY"] = "0"
!python src/train_loras.py --task code --epochs 2 --max-seq-len 768 --batch-size 8 --grad-accum 1 --output-dir /content/drive/MyDrive/llm-serving/adapters_v2


/content

[INIT] Starting QLoRA Training Runner for task: [CODE]
[INIT] Model: Qwen/Qwen2.5-1.5B-Instruct | Output: /content/drive/MyDrive/llm-serving/adapters_v2
[INIT] Loading CUDA & PyTorch libraries (takes ~20-30s on Windows for initial DLL load)...

STARTING QLoRA TRAINING FOR TASK: [CODE]
Base Model: Qwen/Qwen2.5-1.5B-Instruct
LoRA Rank: 16 | Alpha: 32 | Target: q_proj, v_proj
Device: Tesla T4
Loading tokenizer...
config.json: 100% 660/660 [00:00<00:00, 2.24MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 13.9MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 66.1MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 91.0MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 119MB/s]
Precision Mode: FP16
Loading base model in 4-bit precision...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 3.09G/3.09G [00:36<00:00, 85.2MB/s]
Loading weights: 100% 338/338 [00:04<00:00, 82.75it/s] 
generation_config.json: 100% 242/242 [00:00<00:00, 1.17MB

In [6]:
!cd /content/drive/MyDrive/llm-serving/adapters_v2/code_lora && zip -j /content/drive/MyDrive/llm-serving/code_lora_v2.zip adapter_model.safetensors adapter_config.json training_log.json
!ls -la /content/drive/MyDrive/llm-serving/


  adding: adapter_model.safetensors (deflated 8%)
  adding: adapter_config.json (deflated 60%)
  adding: training_log.json (deflated 77%)
total 7890
drwx------ 3 root root    4096 Oct  1 09:00 .
drwx------ 5 root root    4096 Oct  1 08:09 ..
drwx------ 3 root root    4096 Oct  1 08:10 adapters_v2
-rw------- 1 root root 8066553 Oct  1 09:00 code_lora_v2.zip
